In [1]:
# hea_agent_web_fixed.py

from __future__ import annotations

from contextlib import asynccontextmanager
from typing import List, Dict, Any, Optional, Tuple
from pathlib import Path
from datetime import datetime
from math import erf, sqrt, pi
import sqlite3
import json
import uuid
import csv
import re
import math

import numpy as np
from fastapi import FastAPI, HTTPException
from fastapi.responses import HTMLResponse, FileResponse
from pydantic import BaseModel, Field
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import Matern, WhiteKernel, ConstantKernel


DB_PATH = Path("hea_agent_fixed.db")
REPORT_DIR = Path("reports")
REPORT_DIR.mkdir(exist_ok=True)


def now_iso():
    return datetime.utcnow().isoformat() + "Z"


def clean_text(x):
    if x is None:
        return ""
    return re.sub(r"\s+", " ", str(x)).strip()


def chunk_text(text, chunk_size=900, overlap=120):
    text = clean_text(text)
    if not text:
        return []
    chunks = []
    start = 0
    while start < len(text):
        end = min(start + chunk_size, len(text))
        chunks.append(text[start:end])
        if end == len(text):
            break
        start = end - overlap
    return chunks


ELEMENTS = {
    "Al": {"r": 1.432, "vec": 3.0, "tm": 933.47},
    "Co": {"r": 1.251, "vec": 9.0, "tm": 1768.0},
    "Cr": {"r": 1.249, "vec": 6.0, "tm": 2180.0},
    "Fe": {"r": 1.241, "vec": 8.0, "tm": 1811.0},
    "Ni": {"r": 1.246, "vec": 10.0, "tm": 1728.0},
}

R_GAS = 8.314


def normalize_comp(comp):
    total = sum(max(0.0, float(v)) for k, v in comp.items() if k in ELEMENTS)
    if total <= 0:
        raise ValueError("No valid composition.")
    return {k: max(0.0, float(v)) / total for k, v in comp.items() if k in ELEMENTS}


def vec(comp):
    x = normalize_comp(comp)
    return sum(v * ELEMENTS[k]["vec"] for k, v in x.items())


def delta(comp):
    x = normalize_comp(comp)
    rbar = sum(v * ELEMENTS[k]["r"] for k, v in x.items())
    return 100 * math.sqrt(sum(v * (1 - ELEMENTS[k]["r"] / rbar) ** 2 for k, v in x.items()))


class Store:
    def __init__(self, path):
        self.path = path
        self.init_db()

    def conn(self):
        c = sqlite3.connect(self.path, check_same_thread=False)
        c.row_factory = sqlite3.Row
        return c

    def init_db(self):
        c = self.conn()
        cur = c.cursor()
        cur.execute("""
        CREATE TABLE IF NOT EXISTS chunks(
            id TEXT PRIMARY KEY,
            title TEXT,
            text TEXT,
            metadata TEXT,
            created_at TEXT
        )
        """)
        cur.execute("""
        CREATE TABLE IF NOT EXISTS experiments(
            id TEXT PRIMARY KEY,
            params TEXT,
            objective REAL,
            critic TEXT,
            note TEXT,
            created_at TEXT
        )
        """)
        c.commit()
        c.close()

    def add_chunk(self, title, text, metadata):
        c = self.conn()
        c.execute(
            "INSERT INTO chunks VALUES (?, ?, ?, ?, ?)",
            (str(uuid.uuid4()), title, text, json.dumps(metadata), now_iso())
        )
        c.commit()
        c.close()

    def chunks(self):
        c = self.conn()
        rows = c.execute("SELECT * FROM chunks ORDER BY created_at").fetchall()
        c.close()
        return [dict(r) for r in rows]

    def add_experiment(self, params, objective, critic, note):
        c = self.conn()
        c.execute(
            "INSERT INTO experiments VALUES (?, ?, ?, ?, ?, ?)",
            (str(uuid.uuid4()), json.dumps(params), float(objective), json.dumps(critic), note, now_iso())
        )
        c.commit()
        c.close()

    def experiments(self):
        c = self.conn()
        rows = c.execute("SELECT * FROM experiments ORDER BY created_at").fetchall()
        c.close()
        out = []
        for r in rows:
            out.append({
                "id": r["id"],
                "params": json.loads(r["params"]),
                "objective": r["objective"],
                "critic": json.loads(r["critic"]),
                "note": r["note"],
                "created_at": r["created_at"],
            })
        return out


class RetrieverAgent:
    def __init__(self, store):
        self.store = store
        self.vectorizer = TfidfVectorizer(stop_words="english", ngram_range=(1, 2))
        self.matrix = None
        self.rows = []

    def rebuild(self):
        self.rows = self.store.chunks()
        if not self.rows:
            self.matrix = None
            return
        texts = [r["text"] for r in self.rows]
        self.matrix = self.vectorizer.fit_transform(texts)

    def ingest(self, title, text, metadata=None):
        metadata = metadata or {}
        for ch in chunk_text(text):
            self.store.add_chunk(title, ch, metadata)
        self.rebuild()

    def search(self, query, top_k=5):
        if self.matrix is None:
            self.rebuild()
        if self.matrix is None:
            return []
        q = self.vectorizer.transform([query])
        scores = cosine_similarity(q, self.matrix).ravel()
        idxs = np.argsort(scores)[::-1][:top_k]
        return [
            {
                "title": self.rows[i]["title"],
                "text": self.rows[i]["text"],
                "score": float(scores[i]),
                "metadata": json.loads(self.rows[i]["metadata"] or "{}"),
            }
            for i in idxs
        ]


class SummarizerAgent:
    def summarize(self, query, chunks):
        if not chunks:
            return "No literature evidence found. Please ingest literature first."
        lines = [f"Question: {query}", "", "Evidence-backed synthesis:"]
        for i, c in enumerate(chunks, 1):
            lines.append(f"{i}. {c['title']} | score={c['score']:.3f}")
            lines.append(c["text"][:450])
            lines.append("")
        lines.append("Practical HEA takeaway: use retrieved evidence as guidance and confirm with CALPHAD/experiments.")
        return "\n".join(lines)


class CriticAgent:
    def evaluate(self, params):
        comp = {}
        mapping = {"al_pct": "Al", "co_pct": "Co", "cr_pct": "Cr", "fe_pct": "Fe", "ni_pct": "Ni"}
        for k, el in mapping.items():
            if k in params:
                comp[el] = params[k]

        if len(comp) < 2:
            return {"status": "warning", "message": "Need at least 2 composition fields."}

        v = vec(comp)
        d = delta(comp)

        checks = []
        if v >= 8:
            checks.append("VEC suggests FCC-leaning tendency.")
        elif v < 6.87:
            checks.append("VEC suggests BCC-leaning tendency.")
        else:
            checks.append("VEC suggests mixed FCC+BCC possibility.")

        if d <= 6.6:
            checks.append("Atomic size mismatch is favorable for solid solution.")
        else:
            checks.append("High atomic size mismatch; intermetallic risk increases.")

        status = "pass" if d <= 6.6 else "review"

        return {
            "status": status,
            "metrics": {"VEC": v, "delta_percent": d},
            "checks": checks
        }


class PlannerAgent:
    def __init__(self, store):
        self.store = store
        self.param_names = []
        self.bounds = []
        self.X = []
        self.y = []
        self.gp = None
        self.rng = np.random.RandomState(42)
        self.configure_default()

    def configure_default(self):
        params = [
            ("al_pct", 0, 20),
            ("co_pct", 10, 30),
            ("cr_pct", 10, 30),
            ("fe_pct", 10, 30),
            ("ni_pct", 10, 30),
            ("anneal_temp_c", 700, 1200),
            ("anneal_time_h", 0.5, 24),
            ("cooling_rate_c_per_min", 1, 50),
        ]
        self.param_names = [p[0] for p in params]
        self.bounds = [(p[1], p[2]) for p in params]
        kernel = ConstantKernel(1.0) * Matern(length_scale=np.ones(len(params)), nu=2.5) + WhiteKernel()
        self.gp = GaussianProcessRegressor(kernel=kernel, normalize_y=True, random_state=42)

    def fit_if_possible(self):
        if len(self.X) >= 2:
            self.gp.fit(np.array(self.X), np.array(self.y))

    def normal_pdf(self, z):
        return np.exp(-0.5 * z * z) / sqrt(2 * pi)

    def normal_cdf(self, z):
        return 0.5 * (1 + np.vectorize(erf)(z / sqrt(2)))

    def suggest(self, n=3):
        candidates = np.column_stack([
            self.rng.uniform(low, high, 2000)
            for low, high in self.bounds
        ])

        if len(self.X) < 2:
            chosen = candidates[:n]
        else:
            mu, std = self.gp.predict(candidates, return_std=True)
            best = max(self.y)
            std = np.maximum(std, 1e-9)
            z = (mu - best - 0.01) / std
            ei = (mu - best - 0.01) * self.normal_cdf(z) + std * self.normal_pdf(z)
            chosen = candidates[np.argsort(ei)[::-1][:n]]

        return [
            {name: float(val) for name, val in zip(self.param_names, row)}
            for row in chosen
        ]

    def observe(self, params, objective):
        x = [float(params[k]) for k in self.param_names]
        self.X.append(x)
        self.y.append(float(objective))
        self.fit_if_possible()


class ReportAgent:
    def __init__(self, store):
        self.store = store

    def export_csv(self):
        rows = self.store.experiments()
        path = REPORT_DIR / f"hea_experiments_{datetime.utcnow().strftime('%Y%m%d_%H%M%S')}.csv"
        keys = ["al_pct", "co_pct", "cr_pct", "fe_pct", "ni_pct", "anneal_temp_c", "anneal_time_h", "cooling_rate_c_per_min"]
        with open(path, "w", newline="", encoding="utf-8") as f:
            w = csv.writer(f)
            w.writerow(["id", "created_at", "objective", "critic_status", "note"] + keys)
            for r in rows:
                w.writerow(
                    [r["id"], r["created_at"], r["objective"], r["critic"].get("status", ""), r["note"]]
                    + [r["params"].get(k, "") for k in keys]
                )
        return path


class HEASystem:
    def __init__(self):
        self.store = Store(DB_PATH)
        self.retriever = RetrieverAgent(self.store)
        self.summarizer = SummarizerAgent()
        self.critic = CriticAgent()
        self.planner = PlannerAgent(self.store)
        self.reporter = ReportAgent(self.store)

    def propose(self, goal, query, n=3):
        chunks = self.retriever.search(query)
        summary = self.summarizer.summarize(query, chunks)
        suggestions = self.planner.suggest(n)
        reviewed = [{"proposal": s, "critic": self.critic.evaluate(s)} for s in suggestions]
        return {"goal": goal, "literature_summary": summary, "reviewed_proposals": reviewed}

    def record(self, params, objective, note):
        critic = self.critic.evaluate(params)
        self.store.add_experiment(params, objective, critic, note)
        self.planner.observe(params, objective)
        return {"message": "Experiment recorded", "critic": critic}


class LiteratureDocIn(BaseModel):
    title: str
    text: str
    metadata: Optional[Dict[str, Any]] = Field(default_factory=dict)


class IngestRequest(BaseModel):
    documents: List[LiteratureDocIn]


class QueryRequest(BaseModel):
    query: str
    top_k: int = 5


class ProposeRequest(BaseModel):
    research_goal: str
    literature_query: str
    n_suggestions: int = 3


class RecordRequest(BaseModel):
    params: Dict[str, float]
    objective_value: float
    note: str = ""


STATE = {}


@asynccontextmanager
async def lifespan(app: FastAPI):
    STATE["system"] = HEASystem()
    yield
    STATE.clear()


app = FastAPI(title="HEA AI Agent Web App Fixed", lifespan=lifespan)


HTML = """
<!DOCTYPE html>
<html>
<head>
<title>HEA AI Agent</title>
<style>
body{font-family:Arial;background:#f4f7fb;margin:24px;color:#111827}
.wrap{max-width:1200px;margin:auto}
.card{background:white;border-radius:14px;padding:18px;margin-bottom:18px;box-shadow:0 4px 14px rgba(0,0,0,.08)}
.grid{display:grid;grid-template-columns:1fr 1fr;gap:18px}
input,textarea{width:100%;box-sizing:border-box;padding:10px;margin:8px 0;border:1px solid #cbd5e1;border-radius:8px}
button{background:#2563eb;color:white;border:0;padding:10px 14px;border-radius:8px;cursor:pointer}
pre{white-space:pre-wrap;background:#0f172a;color:#e5e7eb;padding:12px;border-radius:10px}
</style>
</head>
<body>
<div class="wrap">
<div class="card">
<h1>HEA Multi-Agent AI Platform</h1>
<p>Web app is running. No FAISS / SentenceTransformer startup blocking.</p>
<p><a href="/docs">API Docs</a> | <a href="/ping">Ping</a> | <a href="/health">Health</a></p>
</div>

<div class="grid">
<div class="card">
<h2>1. Ingest Literature</h2>
<input id="title" value="HEA paper note">
<textarea id="text" rows="7">Increasing Al content can promote BCC or B2 phases in AlCoCrFeNi-type HEAs and may increase hardness. Annealing temperature affects precipitation, grain growth, and phase stability.</textarea>
<button onclick="ingest()">Ingest</button>
<pre id="ingest_out"></pre>
</div>

<div class="card">
<h2>2. Ask Literature</h2>
<input id="query" value="How does Al affect hardness and phase stability in HEAs?">
<button onclick="ask()">Ask</button>
<pre id="ask_out"></pre>
</div>
</div>

<div class="grid">
<div class="card">
<h2>3. Propose Experiments</h2>
<input id="goal" value="Increase hardness while retaining acceptable phase stability">
<input id="lit_query" value="Al addition annealing hardness phase stability HEA">
<input id="n" type="number" value="3">
<button onclick="propose()">Propose</button>
<pre id="prop_out"></pre>
</div>

<div class="card">
<h2>4. Record Experiment</h2>
<input id="al" type="number" value="8">
<input id="co" type="number" value="23">
<input id="cr" type="number" value="22">
<input id="fe" type="number" value="22">
<input id="ni" type="number" value="25">
<input id="temp" type="number" value="950">
<input id="time" type="number" value="6">
<input id="cool" type="number" value="10">
<input id="obj" type="number" value="91.5">
<textarea id="note">Hardness focused trial</textarea>
<button onclick="record()">Record</button>
<pre id="rec_out"></pre>
</div>
</div>

<div class="card">
<h2>Export</h2>
<button onclick="window.open('/export/csv','_blank')">Export CSV</button>
</div>
</div>

<script>
async function ingest(){
 const payload={documents:[{title:document.getElementById('title').value,text:document.getElementById('text').value,metadata:{domain:'HEA'}}]};
 const r=await fetch('/literature/ingest',{method:'POST',headers:{'Content-Type':'application/json'},body:JSON.stringify(payload)});
 document.getElementById('ingest_out').textContent=JSON.stringify(await r.json(),null,2);
}
async function ask(){
 const payload={query:document.getElementById('query').value,top_k:5};
 const r=await fetch('/literature/answer',{method:'POST',headers:{'Content-Type':'application/json'},body:JSON.stringify(payload)});
 document.getElementById('ask_out').textContent=JSON.stringify(await r.json(),null,2);
}
async function propose(){
 const payload={research_goal:document.getElementById('goal').value,literature_query:document.getElementById('lit_query').value,n_suggestions:parseInt(document.getElementById('n').value)};
 const r=await fetch('/agent/propose',{method:'POST',headers:{'Content-Type':'application/json'},body:JSON.stringify(payload)});
 document.getElementById('prop_out').textContent=JSON.stringify(await r.json(),null,2);
}
async function record(){
 const payload={
  params:{
   al_pct:parseFloat(document.getElementById('al').value),
   co_pct:parseFloat(document.getElementById('co').value),
   cr_pct:parseFloat(document.getElementById('cr').value),
   fe_pct:parseFloat(document.getElementById('fe').value),
   ni_pct:parseFloat(document.getElementById('ni').value),
   anneal_temp_c:parseFloat(document.getElementById('temp').value),
   anneal_time_h:parseFloat(document.getElementById('time').value),
   cooling_rate_c_per_min:parseFloat(document.getElementById('cool').value)
  },
  objective_value:parseFloat(document.getElementById('obj').value),
  note:document.getElementById('note').value
 };
 const r=await fetch('/experiments/record',{method:'POST',headers:{'Content-Type':'application/json'},body:JSON.stringify(payload)});
 document.getElementById('rec_out').textContent=JSON.stringify(await r.json(),null,2);
}
</script>
</body>
</html>
"""


@app.get("/ping")
def ping():
    return {"ok": True}


@app.get("/health")
def health():
    s = STATE["system"]
    return {
        "status": "ok",
        "chunks": len(s.store.chunks()),
        "experiments": len(s.store.experiments())
    }


@app.get("/", response_class=HTMLResponse)
def home():
    return HTML


@app.post("/literature/ingest")
def ingest(req: IngestRequest):
    s = STATE["system"]
    for d in req.documents:
        s.retriever.ingest(d.title, d.text, d.metadata)
    return {"message": "Literature ingested", "chunks": len(s.store.chunks())}


@app.post("/literature/answer")
def answer(req: QueryRequest):
    s = STATE["system"]
    chunks = s.retriever.search(req.query, req.top_k)
    return {
        "query": req.query,
        "summary": s.summarizer.summarize(req.query, chunks),
        "chunks": chunks
    }


@app.post("/agent/propose")
def propose(req: ProposeRequest):
    s = STATE["system"]
    return s.propose(req.research_goal, req.literature_query, req.n_suggestions)


@app.post("/experiments/record")
def record(req: RecordRequest):
    s = STATE["system"]
    return s.record(req.params, req.objective_value, req.note)


@app.get("/experiments/history")
def history():
    return {"experiments": STATE["system"].store.experiments()}


@app.get("/export/csv")
def export_csv():
    path = STATE["system"].reporter.export_csv()
    return FileResponse(path, media_type="text/csv", filename=path.name)


if __name__ == "__main__":
    import uvicorn
    import nest_asyncio

    nest_asyncio.apply()

    config = uvicorn.Config(
        app,
        host="127.0.0.1",
        port=5055,
        reload=False,
        log_level="info"
    )

    server = uvicorn.Server(config)
    server.run()

INFO:     Started server process [11332]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://127.0.0.1:5055 (Press CTRL+C to quit)


INFO:     127.0.0.1:49423 - "GET / HTTP/1.1" 200 OK
INFO:     127.0.0.1:49423 - "GET /favicon.ico HTTP/1.1" 404 Not Found
INFO:     127.0.0.1:53273 - "POST /literature/ingest HTTP/1.1" 200 OK
INFO:     127.0.0.1:62779 - "POST /literature/answer HTTP/1.1" 200 OK
INFO:     127.0.0.1:51385 - "POST /agent/propose HTTP/1.1" 200 OK
INFO:     127.0.0.1:51385 - "POST /experiments/record HTTP/1.1" 200 OK
INFO:     127.0.0.1:51385 - "GET /export/csv HTTP/1.1" 200 OK
INFO:     127.0.0.1:55655 - "GET /docs HTTP/1.1" 200 OK
INFO:     127.0.0.1:55655 - "GET /openapi.json HTTP/1.1" 200 OK
INFO:     127.0.0.1:49443 - "POST /literature/ingest HTTP/1.1" 200 OK
INFO:     127.0.0.1:49443 - "POST /literature/ingest HTTP/1.1" 200 OK
INFO:     127.0.0.1:49443 - "POST /literature/answer HTTP/1.1" 200 OK
INFO:     127.0.0.1:53545 - "POST /literature/answer HTTP/1.1" 200 OK


INFO:     Shutting down
INFO:     Waiting for application shutdown.
INFO:     Application shutdown complete.
INFO:     Finished server process [11332]


In [2]:
# hea_agent_property_dashboard.py

from __future__ import annotations

from contextlib import asynccontextmanager
from typing import List, Dict, Any, Optional
from pathlib import Path
from datetime import datetime
from math import erf, sqrt, pi
import sqlite3
import json
import uuid
import csv
import re
import math

import numpy as np

from fastapi import FastAPI
from fastapi.responses import HTMLResponse, FileResponse
from pydantic import BaseModel, Field

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import Matern, WhiteKernel, ConstantKernel
from sklearn.ensemble import RandomForestRegressor, ExtraTreesRegressor, GradientBoostingRegressor
from sklearn.linear_model import Ridge
from sklearn.metrics import r2_score, mean_absolute_error


DB_PATH = Path("hea_agent_property_dashboard.db")
REPORT_DIR = Path("reports")
REPORT_DIR.mkdir(exist_ok=True)


def now_iso():
    return datetime.utcnow().isoformat() + "Z"


def clean_text(x):
    if x is None:
        return ""
    return re.sub(r"\s+", " ", str(x)).strip()


def chunk_text(text, chunk_size=900, overlap=120):
    text = clean_text(text)
    if not text:
        return []
    chunks = []
    start = 0
    while start < len(text):
        end = min(start + chunk_size, len(text))
        chunks.append(text[start:end])
        if end == len(text):
            break
        start = end - overlap
    return chunks


ELEMENTS = {
    "Al": {"r": 1.432, "vec": 3.0, "tm": 933.47},
    "Co": {"r": 1.251, "vec": 9.0, "tm": 1768.0},
    "Cr": {"r": 1.249, "vec": 6.0, "tm": 2180.0},
    "Fe": {"r": 1.241, "vec": 8.0, "tm": 1811.0},
    "Ni": {"r": 1.246, "vec": 10.0, "tm": 1728.0},
}

R_GAS = 8.314


def normalize_comp(comp):
    total = sum(max(0.0, float(v)) for k, v in comp.items() if k in ELEMENTS)
    if total <= 0:
        raise ValueError("No valid composition.")
    return {k: max(0.0, float(v)) / total for k, v in comp.items() if k in ELEMENTS}


def calc_vec(comp):
    x = normalize_comp(comp)
    return sum(v * ELEMENTS[k]["vec"] for k, v in x.items())


def calc_delta(comp):
    x = normalize_comp(comp)
    rbar = sum(v * ELEMENTS[k]["r"] for k, v in x.items())
    return 100 * math.sqrt(sum(v * (1 - ELEMENTS[k]["r"] / rbar) ** 2 for k, v in x.items()))


def calc_entropy(comp):
    x = normalize_comp(comp)
    return -R_GAS * sum(v * math.log(v) for v in x.values() if v > 0)


class Store:
    def __init__(self, path):
        self.path = path
        self.init_db()

    def conn(self):
        c = sqlite3.connect(self.path, check_same_thread=False)
        c.row_factory = sqlite3.Row
        return c

    def init_db(self):
        c = self.conn()
        cur = c.cursor()

        cur.execute("""
        CREATE TABLE IF NOT EXISTS chunks(
            id TEXT PRIMARY KEY,
            title TEXT,
            text TEXT,
            metadata TEXT,
            created_at TEXT
        )
        """)

        cur.execute("""
        CREATE TABLE IF NOT EXISTS experiments(
            id TEXT PRIMARY KEY,
            params TEXT,
            objective REAL,
            critic TEXT,
            note TEXT,
            created_at TEXT
        )
        """)

        c.commit()
        c.close()

    def add_chunk(self, title, text, metadata):
        c = self.conn()
        c.execute(
            "INSERT INTO chunks VALUES (?, ?, ?, ?, ?)",
            (str(uuid.uuid4()), title, text, json.dumps(metadata), now_iso())
        )
        c.commit()
        c.close()

    def chunks(self):
        c = self.conn()
        rows = c.execute("SELECT * FROM chunks ORDER BY created_at").fetchall()
        c.close()
        return [dict(r) for r in rows]

    def add_experiment(self, params, objective, critic, note):
        c = self.conn()
        c.execute(
            "INSERT INTO experiments VALUES (?, ?, ?, ?, ?, ?)",
            (
                str(uuid.uuid4()),
                json.dumps(params),
                float(objective),
                json.dumps(critic),
                note,
                now_iso(),
            )
        )
        c.commit()
        c.close()

    def experiments(self):
        c = self.conn()
        rows = c.execute("SELECT * FROM experiments ORDER BY created_at").fetchall()
        c.close()
        out = []
        for r in rows:
            out.append({
                "id": r["id"],
                "params": json.loads(r["params"]),
                "objective": float(r["objective"]),
                "critic": json.loads(r["critic"]),
                "note": r["note"],
                "created_at": r["created_at"],
            })
        return out


class RetrieverAgent:
    def __init__(self, store):
        self.store = store
        self.vectorizer = TfidfVectorizer(stop_words="english", ngram_range=(1, 2))
        self.matrix = None
        self.rows = []

    def rebuild(self):
        self.rows = self.store.chunks()
        if not self.rows:
            self.matrix = None
            return
        texts = [r["text"] for r in self.rows]
        self.matrix = self.vectorizer.fit_transform(texts)

    def ingest(self, title, text, metadata=None):
        metadata = metadata or {}
        chunks = chunk_text(text)
        for ch in chunks:
            self.store.add_chunk(title, ch, metadata)
        self.rebuild()
        return len(chunks)

    def search(self, query, top_k=5):
        if self.matrix is None:
            self.rebuild()

        if self.matrix is None:
            return []

        q = self.vectorizer.transform([query])
        scores = cosine_similarity(q, self.matrix).ravel()
        idxs = np.argsort(scores)[::-1][:top_k]

        return [
            {
                "title": self.rows[i]["title"],
                "text": self.rows[i]["text"],
                "score": float(scores[i]),
                "metadata": json.loads(self.rows[i]["metadata"] or "{}"),
            }
            for i in idxs
        ]


class SummarizerAgent:
    def summarize(self, query, chunks):
        if not chunks:
            return "No literature evidence found. Please ingest literature first."

        lines = [f"Question: {query}", "", "Evidence-backed synthesis:"]
        for i, c in enumerate(chunks, 1):
            lines.append(f"{i}. {c['title']} | relevance={c['score']:.3f}")
            lines.append(c["text"][:500])
            lines.append("")

        lines.append("Practical HEA takeaway: use the retrieved evidence as guidance, then validate with CALPHAD, microscopy, and experiments.")
        return "\n".join(lines)


class CriticAgent:
    def evaluate(self, params):
        mapping = {
            "al_pct": "Al",
            "co_pct": "Co",
            "cr_pct": "Cr",
            "fe_pct": "Fe",
            "ni_pct": "Ni",
        }

        comp = {}
        for k, el in mapping.items():
            if k in params:
                comp[el] = float(params[k])

        if len(comp) < 2:
            return {
                "status": "warning",
                "message": "Need at least two composition fields."
            }

        v = calc_vec(comp)
        d = calc_delta(comp)
        s = calc_entropy(comp)

        checks = []

        if v >= 8.0:
            checks.append("VEC suggests FCC-leaning tendency.")
        elif v < 6.87:
            checks.append("VEC suggests BCC-leaning tendency.")
        else:
            checks.append("VEC suggests mixed FCC+BCC tendency.")

        if d <= 6.6:
            checks.append("Atomic size mismatch is favorable for solid solution.")
        else:
            checks.append("High atomic size mismatch; intermetallic risk increases.")

        temp = float(params.get("anneal_temp_c", 0))
        if temp < 500 or temp > 1400:
            checks.append("Annealing temperature is outside common first-pass HEA processing range.")

        status = "pass" if d <= 6.6 else "review"

        return {
            "status": status,
            "metrics": {
                "VEC": float(v),
                "delta_percent": float(d),
                "mixing_entropy_J_molK": float(s),
            },
            "checks": checks
        }


class PlannerAgent:
    def __init__(self, store):
        self.store = store
        self.param_names = []
        self.bounds = []
        self.X = []
        self.y = []
        self.gp = None
        self.rng = np.random.RandomState(42)
        self.configure_default()

    def configure_default(self):
        params = [
            ("al_pct", 0, 20),
            ("co_pct", 10, 30),
            ("cr_pct", 10, 30),
            ("fe_pct", 10, 30),
            ("ni_pct", 10, 30),
            ("anneal_temp_c", 700, 1200),
            ("anneal_time_h", 0.5, 24),
            ("cooling_rate_c_per_min", 1, 50),
        ]

        self.param_names = [p[0] for p in params]
        self.bounds = [(p[1], p[2]) for p in params]

        kernel = (
            ConstantKernel(1.0)
            * Matern(length_scale=np.ones(len(params)), nu=2.5)
            + WhiteKernel()
        )

        self.gp = GaussianProcessRegressor(
            kernel=kernel,
            normalize_y=True,
            random_state=42
        )

    def fit_if_possible(self):
        if len(self.X) >= 2:
            self.gp.fit(np.array(self.X), np.array(self.y))

    def normal_pdf(self, z):
        return np.exp(-0.5 * z * z) / sqrt(2 * pi)

    def normal_cdf(self, z):
        return 0.5 * (1 + np.vectorize(erf)(z / sqrt(2)))

    def suggest(self, n=3):
        candidates = np.column_stack([
            self.rng.uniform(low, high, 2000)
            for low, high in self.bounds
        ])

        if len(self.X) < 2:
            chosen = candidates[:n]
        else:
            mu, std = self.gp.predict(candidates, return_std=True)
            best = max(self.y)
            std = np.maximum(std, 1e-9)
            z = (mu - best - 0.01) / std
            ei = (mu - best - 0.01) * self.normal_cdf(z) + std * self.normal_pdf(z)
            chosen = candidates[np.argsort(ei)[::-1][:n]]

        return [
            {name: float(val) for name, val in zip(self.param_names, row)}
            for row in chosen
        ]

    def observe(self, params, objective):
        x = [float(params[k]) for k in self.param_names]
        self.X.append(x)
        self.y.append(float(objective))
        self.fit_if_possible()


class PropertyPredictorAgent:
    def __init__(self, store):
        self.store = store
        self.models = {
            "RandomForest": RandomForestRegressor(n_estimators=200, random_state=42),
            "ExtraTrees": ExtraTreesRegressor(n_estimators=200, random_state=42),
            "GradientBoosting": GradientBoostingRegressor(random_state=42),
            "Ridge": Ridge()
        }
        self.best_model_name = None
        self.best_model = None
        self.metrics = {}

        self.feature_names = [
            "al_pct",
            "co_pct",
            "cr_pct",
            "fe_pct",
            "ni_pct",
            "anneal_temp_c",
            "anneal_time_h",
            "cooling_rate_c_per_min",
            "VEC",
            "delta_percent",
            "mixing_entropy_J_molK",
        ]

    def make_features(self, params):
        comp = {
            "Al": params.get("al_pct", 0),
            "Co": params.get("co_pct", 0),
            "Cr": params.get("cr_pct", 0),
            "Fe": params.get("fe_pct", 0),
            "Ni": params.get("ni_pct", 0),
        }

        try:
            v = calc_vec(comp)
            d = calc_delta(comp)
            s = calc_entropy(comp)
        except Exception:
            v, d, s = 0.0, 0.0, 0.0

        return [
            float(params.get("al_pct", 0)),
            float(params.get("co_pct", 0)),
            float(params.get("cr_pct", 0)),
            float(params.get("fe_pct", 0)),
            float(params.get("ni_pct", 0)),
            float(params.get("anneal_temp_c", 0)),
            float(params.get("anneal_time_h", 0)),
            float(params.get("cooling_rate_c_per_min", 0)),
            float(v),
            float(d),
            float(s),
        ]

    def train(self):
        rows = self.store.experiments()

        if len(rows) < 3:
            return {
                "error": "Need at least 3 recorded experiments to train models.",
                "available_rows": len(rows)
            }

        X = np.array([self.make_features(r["params"]) for r in rows], dtype=float)
        y = np.array([float(r["objective"]) for r in rows], dtype=float)

        self.metrics = {}
        best_score = -1e18

        for name, model in self.models.items():
            model.fit(X, y)
            pred = model.predict(X)

            r2 = r2_score(y, pred) if len(set(y)) > 1 else 0.0
            mae = mean_absolute_error(y, pred)

            self.metrics[name] = {
                "r2_train": float(r2),
                "mae_train": float(mae)
            }

            score = r2 - 0.01 * mae

            if score > best_score:
                best_score = score
                self.best_model_name = name
                self.best_model = model

        return {
            "message": "Models trained successfully.",
            "rows": len(rows),
            "best_model": self.best_model_name,
            "metrics": self.metrics,
            "features": self.feature_names
        }

    def predict(self, params):
        if self.best_model is None:
            train_result = self.train()
            if "error" in train_result:
                return train_result

        x = np.array([self.make_features(params)], dtype=float)

        model_preds = {}
        for name, model in self.models.items():
            try:
                model_preds[name] = float(model.predict(x)[0])
            except Exception:
                model_preds[name] = None

        valid_preds = [v for v in model_preds.values() if v is not None]
        uncertainty = float(np.std(valid_preds)) if valid_preds else None

        return {
            "predicted_property_score": float(self.best_model.predict(x)[0]),
            "best_model": self.best_model_name,
            "model_predictions": model_preds,
            "uncertainty_std": uncertainty,
            "model_metrics": self.metrics,
            "features_used": dict(zip(self.feature_names, self.make_features(params)))
        }


class ReportAgent:
    def __init__(self, store):
        self.store = store

    def export_csv(self):
        rows = self.store.experiments()
        path = REPORT_DIR / f"hea_experiments_{datetime.utcnow().strftime('%Y%m%d_%H%M%S')}.csv"

        keys = [
            "al_pct",
            "co_pct",
            "cr_pct",
            "fe_pct",
            "ni_pct",
            "anneal_temp_c",
            "anneal_time_h",
            "cooling_rate_c_per_min"
        ]

        with open(path, "w", newline="", encoding="utf-8") as f:
            w = csv.writer(f)
            w.writerow(["id", "created_at", "objective", "critic_status", "note"] + keys)

            for r in rows:
                w.writerow(
                    [
                        r["id"],
                        r["created_at"],
                        r["objective"],
                        r["critic"].get("status", ""),
                        r["note"]
                    ]
                    + [r["params"].get(k, "") for k in keys]
                )

        return path


class HEASystem:
    def __init__(self):
        self.store = Store(DB_PATH)
        self.retriever = RetrieverAgent(self.store)
        self.summarizer = SummarizerAgent()
        self.critic = CriticAgent()
        self.planner = PlannerAgent(self.store)
        self.predictor = PropertyPredictorAgent(self.store)
        self.reporter = ReportAgent(self.store)

    def propose(self, goal, query, n=3):
        chunks = self.retriever.search(query)
        summary = self.summarizer.summarize(query, chunks)
        suggestions = self.planner.suggest(n)
        reviewed = [{"proposal": s, "critic": self.critic.evaluate(s)} for s in suggestions]

        return {
            "goal": goal,
            "literature_summary": summary,
            "reviewed_proposals": reviewed
        }

    def record(self, params, objective, note):
        critic = self.critic.evaluate(params)
        self.store.add_experiment(params, objective, critic, note)
        self.planner.observe(params, objective)

        return {
            "message": "Experiment recorded.",
            "critic": critic,
            "total_experiments": len(self.store.experiments())
        }


class LiteratureDocIn(BaseModel):
    title: str
    text: str
    metadata: Optional[Dict[str, Any]] = Field(default_factory=dict)


class IngestRequest(BaseModel):
    documents: List[LiteratureDocIn]


class QueryRequest(BaseModel):
    query: str
    top_k: int = 5


class ProposeRequest(BaseModel):
    research_goal: str
    literature_query: str
    n_suggestions: int = 3


class RecordRequest(BaseModel):
    params: Dict[str, float]
    objective_value: float
    note: str = ""


class PredictRequest(BaseModel):
    params: Dict[str, float]


STATE = {}


@asynccontextmanager
async def lifespan(app: FastAPI):
    STATE["system"] = HEASystem()
    yield
    STATE.clear()


app = FastAPI(title="HEA Property Prediction AI Agent", lifespan=lifespan)


HTML = """
<!DOCTYPE html>
<html>
<head>
<title>HEA Property Prediction AI Agent</title>
<style>
body{font-family:Arial;background:#f4f7fb;margin:24px;color:#111827}
.wrap{max-width:1250px;margin:auto}
.card{background:white;border-radius:14px;padding:18px;margin-bottom:18px;box-shadow:0 4px 14px rgba(0,0,0,.08)}
.grid{display:grid;grid-template-columns:1fr 1fr;gap:18px}
input,textarea{width:100%;box-sizing:border-box;padding:10px;margin:8px 0;border:1px solid #cbd5e1;border-radius:8px}
button{background:#2563eb;color:white;border:0;padding:10px 14px;border-radius:8px;cursor:pointer;margin:4px}
pre{white-space:pre-wrap;background:#0f172a;color:#e5e7eb;padding:12px;border-radius:10px;max-height:450px;overflow:auto}
.badge{display:inline-block;background:#e0ecff;color:#1d4ed8;padding:6px 10px;border-radius:999px;margin:3px}
</style>
</head>
<body>
<div class="wrap">

<div class="card">
<h1>HEA Multi-Agent Property Prediction Dashboard</h1>
<span class="badge">RAG Literature</span>
<span class="badge">Bayesian Proposal</span>
<span class="badge">HEA Critic</span>
<span class="badge">Property Prediction</span>
<span class="badge">Model Comparison</span>
<p><a href="/docs">API Docs</a> | <a href="/ping">Ping</a> | <a href="/health">Health</a></p>
</div>

<div class="grid">
<div class="card">
<h2>1. Ingest Literature</h2>
<input id="title" value="HEA literature note">
<textarea id="text" rows="7">Increasing Al content can promote BCC or B2 phases in AlCoCrFeNi-type HEAs and may increase hardness. Annealing temperature affects precipitation, grain growth, and phase stability.</textarea>
<button onclick="ingest()">Ingest Literature</button>
<pre id="ingest_out"></pre>
</div>

<div class="card">
<h2>2. Ask Literature</h2>
<input id="query" value="How does Al affect hardness and phase stability in HEAs?">
<button onclick="ask()">Ask RAG</button>
<pre id="ask_out"></pre>
</div>
</div>

<div class="grid">
<div class="card">
<h2>3. Propose Next Experiments</h2>
<input id="goal" value="Increase hardness while retaining acceptable phase stability">
<input id="lit_query" value="Al addition annealing hardness phase stability HEA">
<input id="n" type="number" value="3">
<button onclick="propose()">Propose Experiments</button>
<pre id="prop_out"></pre>
</div>

<div class="card">
<h2>4. Record Experiment</h2>
<input id="al" type="number" value="8" placeholder="Al at.%">
<input id="co" type="number" value="23" placeholder="Co at.%">
<input id="cr" type="number" value="22" placeholder="Cr at.%">
<input id="fe" type="number" value="22" placeholder="Fe at.%">
<input id="ni" type="number" value="25" placeholder="Ni at.%">
<input id="temp" type="number" value="950" placeholder="Anneal temp C">
<input id="time" type="number" value="6" placeholder="Anneal time h">
<input id="cool" type="number" value="10" placeholder="Cooling rate C/min">
<input id="obj" type="number" value="91.5" placeholder="Measured property/objective">
<textarea id="note">Hardness focused trial</textarea>
<button onclick="record()">Record Experiment</button>
<pre id="rec_out"></pre>
</div>
</div>

<div class="card">
<h2>5. Property Prediction Dashboard</h2>
<p>Record at least 3 experiments, then train and predict property score.</p>
<div class="grid">
<div>
<input id="pal" type="number" value="10" placeholder="Al at.%">
<input id="pco" type="number" value="22" placeholder="Co at.%">
<input id="pcr" type="number" value="22" placeholder="Cr at.%">
<input id="pfe" type="number" value="22" placeholder="Fe at.%">
<input id="pni" type="number" value="24" placeholder="Ni at.%">
</div>
<div>
<input id="ptemp" type="number" value="900" placeholder="Anneal temp C">
<input id="ptime" type="number" value="5" placeholder="Anneal time h">
<input id="pcool" type="number" value="12" placeholder="Cooling rate">
<button onclick="trainModels()">Train Models</button>
<button onclick="predictProperty()">Predict Property</button>
<button onclick="loadHistory()">Show History</button>
</div>
</div>
<pre id="pred_out"></pre>
</div>

<div class="card">
<h2>6. Export</h2>
<button onclick="window.open('/export/csv','_blank')">Export CSV</button>
</div>

</div>

<script>
function recordParams(prefix){
 return {
   al_pct:parseFloat(document.getElementById(prefix+'al').value),
   co_pct:parseFloat(document.getElementById(prefix+'co').value),
   cr_pct:parseFloat(document.getElementById(prefix+'cr').value),
   fe_pct:parseFloat(document.getElementById(prefix+'fe').value),
   ni_pct:parseFloat(document.getElementById(prefix+'ni').value),
   anneal_temp_c:parseFloat(document.getElementById(prefix+'temp').value),
   anneal_time_h:parseFloat(document.getElementById(prefix+'time').value),
   cooling_rate_c_per_min:parseFloat(document.getElementById(prefix+'cool').value)
 };
}

async function ingest(){
 const payload={documents:[{title:document.getElementById('title').value,text:document.getElementById('text').value,metadata:{domain:'HEA'}}]};
 const r=await fetch('/literature/ingest',{method:'POST',headers:{'Content-Type':'application/json'},body:JSON.stringify(payload)});
 document.getElementById('ingest_out').textContent=JSON.stringify(await r.json(),null,2);
}

async function ask(){
 const payload={query:document.getElementById('query').value,top_k:5};
 const r=await fetch('/literature/answer',{method:'POST',headers:{'Content-Type':'application/json'},body:JSON.stringify(payload)});
 document.getElementById('ask_out').textContent=JSON.stringify(await r.json(),null,2);
}

async function propose(){
 const payload={research_goal:document.getElementById('goal').value,literature_query:document.getElementById('lit_query').value,n_suggestions:parseInt(document.getElementById('n').value)};
 const r=await fetch('/agent/propose',{method:'POST',headers:{'Content-Type':'application/json'},body:JSON.stringify(payload)});
 document.getElementById('prop_out').textContent=JSON.stringify(await r.json(),null,2);
}

async function record(){
 const payload={
  params:recordParams(''),
  objective_value:parseFloat(document.getElementById('obj').value),
  note:document.getElementById('note').value
 };
 const r=await fetch('/experiments/record',{method:'POST',headers:{'Content-Type':'application/json'},body:JSON.stringify(payload)});
 document.getElementById('rec_out').textContent=JSON.stringify(await r.json(),null,2);
}

async function trainModels(){
 const r=await fetch('/models/train',{method:'POST'});
 document.getElementById('pred_out').textContent=JSON.stringify(await r.json(),null,2);
}

async function predictProperty(){
 const payload={params:recordParams('p')};
 const r=await fetch('/predict/property',{method:'POST',headers:{'Content-Type':'application/json'},body:JSON.stringify(payload)});
 document.getElementById('pred_out').textContent=JSON.stringify(await r.json(),null,2);
}

async function loadHistory(){
 const r=await fetch('/experiments/history');
 document.getElementById('pred_out').textContent=JSON.stringify(await r.json(),null,2);
}
</script>
</body>
</html>
"""


@app.get("/ping")
def ping():
    return {"ok": True}


@app.get("/health")
def health():
    s = STATE["system"]
    return {
        "status": "ok",
        "chunks": len(s.store.chunks()),
        "experiments": len(s.store.experiments())
    }


@app.get("/", response_class=HTMLResponse)
def home():
    return HTML


@app.post("/literature/ingest")
def ingest(req: IngestRequest):
    s = STATE["system"]
    total = 0
    for d in req.documents:
        total += s.retriever.ingest(d.title, d.text, d.metadata)
    return {"message": "Literature ingested.", "added_chunks": total, "total_chunks": len(s.store.chunks())}


@app.post("/literature/answer")
def answer(req: QueryRequest):
    s = STATE["system"]
    chunks = s.retriever.search(req.query, req.top_k)
    return {
        "query": req.query,
        "summary": s.summarizer.summarize(req.query, chunks),
        "chunks": chunks
    }


@app.post("/agent/propose")
def propose(req: ProposeRequest):
    s = STATE["system"]
    return s.propose(req.research_goal, req.literature_query, req.n_suggestions)


@app.post("/experiments/record")
def record(req: RecordRequest):
    s = STATE["system"]
    return s.record(req.params, req.objective_value, req.note)


@app.get("/experiments/history")
def history():
    return {"experiments": STATE["system"].store.experiments()}


@app.post("/models/train")
def train_models():
    return STATE["system"].predictor.train()


@app.post("/predict/property")
def predict_property(req: PredictRequest):
    s = STATE["system"]
    return {
        "input": req.params,
        "critic": s.critic.evaluate(req.params),
        "prediction": s.predictor.predict(req.params)
    }


@app.get("/export/csv")
def export_csv():
    path = STATE["system"].reporter.export_csv()
    return FileResponse(path, media_type="text/csv", filename=path.name)


if __name__ == "__main__":
    import uvicorn

    try:
        import nest_asyncio
        nest_asyncio.apply()
    except Exception:
        pass

    uvicorn.run(
        app,
        host="127.0.0.1",
        port=5057,
        reload=False
    )

INFO:     Started server process [11332]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://127.0.0.1:5057 (Press CTRL+C to quit)


INFO:     127.0.0.1:61076 - "GET / HTTP/1.1" 200 OK
INFO:     127.0.0.1:61076 - "GET /favicon.ico HTTP/1.1" 404 Not Found
INFO:     127.0.0.1:52047 - "POST /literature/ingest HTTP/1.1" 200 OK
INFO:     127.0.0.1:52047 - "POST /literature/answer HTTP/1.1" 200 OK
INFO:     127.0.0.1:52047 - "POST /agent/propose HTTP/1.1" 200 OK
INFO:     127.0.0.1:52047 - "POST /experiments/record HTTP/1.1" 200 OK
INFO:     127.0.0.1:52047 - "POST /models/train HTTP/1.1" 200 OK
INFO:     127.0.0.1:49890 - "POST /agent/propose HTTP/1.1" 200 OK
INFO:     127.0.0.1:55508 - "POST /experiments/record HTTP/1.1" 200 OK


C:\Users\GOWREESWARI\anaconda3\anaconda\lib\site-packages\sklearn\gaussian_process\kernels.py:420: ConvergenceWarning: The optimal value found for dimension 0 of parameter k1__k1__constant_value is close to the specified lower bound 1e-05. Decreasing the bound and calling fit again may find a better value.
  warnings.warn(
C:\Users\GOWREESWARI\anaconda3\anaconda\lib\site-packages\sklearn\gaussian_process\kernels.py:420: ConvergenceWarning: The optimal value found for dimension 0 of parameter k2__noise_level is close to the specified lower bound 1e-05. Decreasing the bound and calling fit again may find a better value.
  warnings.warn(


INFO:     127.0.0.1:55508 - "POST /experiments/record HTTP/1.1" 200 OK


C:\Users\GOWREESWARI\anaconda3\anaconda\lib\site-packages\sklearn\gaussian_process\kernels.py:420: ConvergenceWarning: The optimal value found for dimension 0 of parameter k1__k1__constant_value is close to the specified lower bound 1e-05. Decreasing the bound and calling fit again may find a better value.
  warnings.warn(
C:\Users\GOWREESWARI\anaconda3\anaconda\lib\site-packages\sklearn\gaussian_process\kernels.py:420: ConvergenceWarning: The optimal value found for dimension 0 of parameter k2__noise_level is close to the specified lower bound 1e-05. Decreasing the bound and calling fit again may find a better value.
  warnings.warn(


...

C:\Users\GOWREESWARI\anaconda3\anaconda\lib\site-packages\sklearn\gaussian_process\kernels.py:420: ConvergenceWarning: The optimal value found for dimension 0 of parameter k1__k1__constant_value is close to the specified lower bound 1e-05. Decreasing the bound and calling fit again may find a better value.
  warnings.warn(
C:\Users\GOWREESWARI\anaconda3\anaconda\lib\site-packages\sklearn\gaussian_process\kernels.py:420: ConvergenceWarning: The optimal value found for dimension 0 of parameter k2__noise_level is close to the specified lower bound 1e-05. Decreasing the bound and calling fit again may find a better value.
  warnings.warn(
C:\Users\GOWREESWARI\anaconda3\anaconda\lib\site-packages\sklearn\gaussian_process\kernels.py:420: ConvergenceWarning: The optimal value found for dimension 0 of parameter k1__k1__constant_value is close to the specified lower bound 1e-05. Decreasing the bound and calling fit again may find a better value.
  warnings.warn(
C:\Users\GOWREESWARI\anaconda3\a

INFO:     127.0.0.1:55508 - "POST /experiments/record HTTP/1.1" 200 OK
INFO:     127.0.0.1:55508 - "POST /models/train HTTP/1.1" 200 OK
INFO:     127.0.0.1:56266 - "POST /predict/property HTTP/1.1" 200 OK
INFO:     127.0.0.1:56266 - "GET /experiments/history HTTP/1.1" 200 OK
INFO:     127.0.0.1:60354 - "GET / HTTP/1.1" 200 OK
INFO:     127.0.0.1:58838 - "POST /literature/ingest HTTP/1.1" 200 OK
INFO:     127.0.0.1:58838 - "POST /literature/answer HTTP/1.1" 200 OK
INFO:     127.0.0.1:53091 - "POST /agent/propose HTTP/1.1" 200 OK
INFO:     127.0.0.1:53091 - "POST /experiments/record HTTP/1.1" 200 OK


C:\Users\GOWREESWARI\anaconda3\anaconda\lib\site-packages\sklearn\gaussian_process\kernels.py:420: ConvergenceWarning: The optimal value found for dimension 0 of parameter k1__k1__constant_value is close to the specified lower bound 1e-05. Decreasing the bound and calling fit again may find a better value.
  warnings.warn(
C:\Users\GOWREESWARI\anaconda3\anaconda\lib\site-packages\sklearn\gaussian_process\kernels.py:420: ConvergenceWarning: The optimal value found for dimension 0 of parameter k2__noise_level is close to the specified lower bound 1e-05. Decreasing the bound and calling fit again may find a better value.
  warnings.warn(


INFO:     127.0.0.1:56950 - "POST /models/train HTTP/1.1" 200 OK
INFO:     127.0.0.1:56950 - "POST /models/train HTTP/1.1" 200 OK
INFO:     127.0.0.1:56950 - "POST /predict/property HTTP/1.1" 200 OK


INFO:     Shutting down
INFO:     Waiting for application shutdown.
INFO:     Application shutdown complete.
INFO:     Finished server process [11332]
